In [ ]:
import fitz  # PyMuPDF
import numpy as np
from pathlib import Path
import re
from anthropic import RateLimitError
from openai import OpenAI
from anthropic import Anthropic
from collections import Counter
from sentence_transformers import SentenceTransformer

anthropic_client = Anthropic()
openai_client = OpenAI()

import logging
from pathlib import Path

import spacy

# ─────────────────────────────────────────────────────────────────────────────
# Logging — configured once at module level
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)


# ─────────────────────────────────────────────────────────────────────────────
# spaCy — loaded once at module level, reused across all files
# ─────────────────────────────────────────────────────────────────────────────
nlp = spacy.load("en_core_web_sm")
nlp.disable_pipes(["ner", "tagger"])   # only the parser is needed for sentence splitting

# Reference section headings — checked case-insensitively during extraction
REFERENCE_HEADINGS = {"references", "bibliography", "works cited"}


# ─────────────────────────────────────────────────────────────────────────────
# Sentence embedding model — loaded once, reused across all semantic refinement calls
# ─────────────────────────────────────────────────────────────────────────────
_embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Reference section headings — checked case-insensitively during extraction
REFERENCE_HEADINGS = {"references", "bibliography", "works cited"}

# A block's font size must exceed body size by this factor to be a heading candidate
HEADING_SIZE_THRESHOLD = 1.15

# Blocks with fewer than this many sentences are merged into their neighbour
MIN_BLOCK_SENTENCES = 3

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — PDF Text Extraction
# ─────────────────────────────────────────────────────────────────────────────
def extract_text_from_pdf(pdf_path: str) -> list[dict] | None:
    """
    Extracts and cleans text from each page of a PDF.

    Layout cleaning applied at extraction time (PyMuPDF's responsibility):
      - Strips headers and footers using bounding box Y position
      - Drops table/figure noise (short numeric-only blocks)
      - Rejoins hyphenated line breaks from two-column layouts
      - Joins stray mid-sentence newlines where next line starts lowercase
      - Detects and discards the references/bibliography section entirely

    Returns None if no extractable text is found (fully scanned/image PDF).
    """
    doc = fitz.open(pdf_path)
    pages = []
    empty_pages = []
    references_reached = False   # flag — once set, all subsequent blocks are dropped

    for i, page in enumerate(doc):
        if references_reached:
            break                # no point reading further pages once references start

        page_height = page.rect.height
        blocks = page.get_text("blocks")   # each block: (x0, y0, x1, y1, text, ...)
        lines = []

        for block in blocks:
            block_text = block[4].strip()
            block_y_top = block[1]
            block_y_bottom = block[3]

            # ── References section detection ──────────────────────────────────
            # Check the first line of each block against known reference headings.
            # Once found, set the flag and stop processing entirely.
            first_line = block_text.split("\n")[0].strip().lower()
            if first_line in REFERENCE_HEADINGS:
                logging.info(f"'{pdf_path}' — references section detected at page {i + 1}, discarding tail.")
                references_reached = True
                break

            # ── Strip headers and footers ─────────────────────────────────────
            # Blocks in top 7% or bottom 7% of the page are headers/footers.
            if block_y_top < page_height * 0.07:
                logging.debug(f"Stripped header block: '{block_text[:50]}'")
                continue
            if block_y_bottom > page_height * 0.93:
                logging.debug(f"Stripped footer block: '{block_text[:50]}'")
                continue

            # ── Drop table and figure noise ───────────────────────────────────
            # Short blocks that are purely numeric/punctuation with no real words.
            if re.fullmatch(r'[\d\s\.\,\%\-]+', block_text) and len(block_text) < 40:
                logging.debug(f"Stripped table noise: '{block_text[:50]}'")
                continue

            # Split block into individual lines for hyphenation and newline fixing
            lines.extend(block_text.split("\n"))

        if references_reached:
            break

        # ── Rejoin hyphenated line breaks ─────────────────────────────────────
        # "large-\nscale" → "large-scale"
        # Strip the trailing hyphen and attach the next line directly (no space).
        rejoined = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            if rejoined and rejoined[-1].endswith("-"):
                rejoined[-1] = rejoined[-1][:-1] + line
            else:
                rejoined.append(line)

        # ── Join stray mid-sentence newlines ──────────────────────────────────
        # If a line doesn't end with sentence-closing punctuation and the next
        # line starts with a lowercase letter, it's a mid-sentence wrap — join
        # with a space. If the next line starts with a capital, leave separate.
        cleaned_lines = []
        for j, line in enumerate(rejoined):
            if (
                cleaned_lines
                and not cleaned_lines[-1][-1] in ".!?"
                and line
                and line[0].islower()
            ):
                cleaned_lines[-1] = cleaned_lines[-1] + " " + line
            else:
                cleaned_lines.append(line)

        text = " ".join(cleaned_lines).strip()

        if text:
            pages.append({"page_num": i + 1, "text": text})
        else:
            empty_pages.append(i + 1)

    doc.close()

    if empty_pages:
        logging.warning(f"'{pdf_path}' — pages with no extractable text: {empty_pages}")

    if not pages:
        logging.error(
            f"'{pdf_path}' — fully scanned/image-based PDF, no text extracted. "
            "Consider OCR (e.g. pytesseract or AWS Textract)."
        )
        return None

    return pages

# Step 1 — detect_structure_boundaries()

```
Change extraction mode from get_text("blocks") to get_text("dict")
  → this gives you font size and font flags per span

For each block on each page:
  → collect the dominant font size of that block
  → collect bold/italic flags

After scanning all pages:
  → compute the body font size (most frequently occurring size)
  → anything with font size > body size by a threshold = heading candidate
  → also flag blocks where ALL spans are bold

For each heading candidate:
  → check it is NOT in the top 7% / bottom 7% zone (skip if it is)
  → check it is NOT a repeat across pages (that's a running header)
  → check it contains at least one real word (not just a number or symbol)
  → if all checks pass → mark this sentence index as a HARD boundary

References/bibliography headings:
  → already handled, keep existing logic, just also register as a hard boundary
  → so the structure detector and the references stopper share the same boundary list
```

# Step 2 — build_coarse_blocks()
```
Start with the full sentence list from split_into_sentences()

Walk through sentences:
  → if current sentence index is a HARD boundary from Step 1:
      → close current block, start a new one
  → else:
      → accumulate sentence into current block

After all sentences are walked:
  → you have N coarse blocks, each aligned to a real section boundary

For each coarse block:
  → estimate its token size using the existing len // 4 method
      BUT also count non-ASCII characters separately and weight them higher
      (handles equations and Greek symbols)
  
  → if block is under max_chunk_tokens → mark as FINAL, no further splitting needed
  → if block is over max_chunk_tokens → mark as NEEDS_SEMANTIC_SPLIT
  → if block is under a minimum threshold (e.g. < 3 sentences):
      → merge it into the previous block instead of keeping it alone
```

# Step 3 — semantic_refine_large_blocks()
```
Only called on blocks marked NEEDS_SEMANTIC_SPLIT

For each oversized block:
  → strip any page markers from sentence text before embedding
     (keep a parallel list mapping sentence index → page number)
  → group sentences into sliding windows of size W (e.g. 5 sentences)
     → embed each WINDOW not each individual sentence
     → this reduces embedding calls from N to N/5

  → compute cosine similarity between each consecutive window pair
  → find the local minima in the similarity curve
     → these are candidate semantic boundaries

  → rank candidates by how deep their similarity drop is
  → accept only candidates where the drop exceeds a threshold
     → this avoids cutting on minor topic wobbles

  → from accepted candidates, pick the one closest to the halfway 
     point of the block first, then recurse on each half if still oversized
     → this prevents runaway splitting into too-small pieces

After splitting:
  → re-attach the page references from the parallel page list
  → check resulting sub-blocks against minimum size threshold again
     → merge tiny leftovers into neighbour
```

# Step 4 — finalize_chunks_with_overlap()
```
At this point you have a flat list of refined blocks, all within token bounds

Walk through blocks in order:
  → assign chunk_id sequentially
  → for overlap:
      → take the last overlap_sentences from the PREVIOUS block
      → prepend them to the CURRENT block's sentence list
      → do NOT re-embed these overlap sentences, just carry the text
      → page_start stays as the first non-overlap sentence's page
        (overlap sentences belong to the previous chunk's page range)

Final chunk output shape stays identical to current code:
  → chunk_id, text, page_start, page_end, sentences
  → add one new field: split_method ("structure" or "semantic")
     → useful for debugging and evaluating which path fired
```

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Sentence Splitting
# ─────────────────────────────────────────────────────────────────────────────

def split_into_sentences(text: str, max_sentence_chars: int = 2048) -> list[str]:
    """
    Splits cleaned text into sentences using spaCy's dependency parser.

    spaCy handles linguistic ambiguity that regex and NLTK cannot:
      - Abbreviations: Dr., et al., Fig., U.S.A.
      - Inline citations: (Smith et al., 2019)
      - Decimal numbers: 96.5, λ = 0.01
      - Complex punctuation in academic text

    Fallback: any sentence exceeding max_sentence_chars is split at the
    nearest whitespace to its midpoint — handles no-punctuation blocks
    that spaCy returns as one oversized unit.

    Layout problems (hyphenation, headers, footers, cross-page boundaries)
    are already cleaned upstream before this function is called.
    """
    doc = nlp(text)
    sentences = []

    for sent in doc.sents:
        s = sent.text.strip()

        if not s or not re.search(r'[a-zA-Z0-9]', s):
            continue   # filter empty or symbol-only fragments

        # ── Oversized sentence fallback ───────────────────────────────────────
        # If spaCy returns a giant block with no punctuation, split it at the
        # nearest whitespace to the midpoint rather than keeping one huge unit.
        if len(s) > max_sentence_chars:
            midpoint = len(s) // 2
            split_at = s.rfind(" ", 0, midpoint)     # nearest space before midpoint
            if split_at == -1:
                split_at = s.find(" ", midpoint)      # fallback: nearest space after
            if split_at != -1:
                left = s[:split_at].strip()
                right = s[split_at:].strip()
                if left:
                    sentences.append(left)
                if right:
                    sentences.append(right)
            else:
                sentences.append(s)   # no whitespace at all — keep as-is
        else:
            sentences.append(s)

    return sentences



In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — OpenAI Embeddings with Batching + Token Guard
# ─────────────────────────────────────────────────────────────────────────────

def get_embeddings(
    texts: list[str],
    model: str = "text-embedding-3-small",
    batch_size: int = 500,
) -> list[list[float]]:
    """
    Gets embeddings from OpenAI in safe batches.

    Fixes applied:
      - Batch size of 500 stays well under the 2048 input limit per request
      - Truncates any single text exceeding ~8000 tokens (1 token ≈ 4 chars)
        to prevent BadRequestError on long academic sentences / footnotes
      - Tracks truncated texts so you know which sentences were cut
    """
    MAX_CHARS = 32000  # ~8000 tokens — OpenAI's per-input hard limit

    cleaned = []
    truncated_count = 0
    for t in texts:
        t = t.replace("\n", " ")
        if len(t) > MAX_CHARS:
            t = t[:MAX_CHARS]
            truncated_count += 1
        cleaned.append(t)

    if truncated_count:
        print(f"  ⚠️  {truncated_count} sentences truncated to fit OpenAI token limit")

    all_embeddings = []
    total_batches = -(-len(cleaned) // batch_size)  # ceiling division

    for i in range(0, len(cleaned), batch_size):
        batch = cleaned[i : i + batch_size]
        batch_num = i // batch_size + 1
        print(f"  Embedding batch {batch_num}/{total_batches} ({len(batch)} sentences)…")
        response = openai_client.embeddings.create(input=batch, model=model)
        all_embeddings.extend([item.embedding for item in response.data])

    return all_embeddings

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Cosine Similarity
# ─────────────────────────────────────────────────────────────────────────────

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """
    Computes cosine similarity between two embedding vectors.

    Fixes applied:
      - Explicit zero-vector guard: returns 0.0 instead of NaN
        (a near-zero embedding means the sentence had no real content,
         treating it as fully dissimilar forces a chunk boundary — safer)
    """
    a, b = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < 1e-10:
        return 0.0  # treat as completely dissimilar → force chunk break
    return float(np.dot(a, b) / denom)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Semantic Chunking
# ─────────────────────────────────────────────────────────────────────────────

def semantic_chunk_pdf(
    pdf_path: str,
    max_chunk_tokens: int = 512,
    overlap_sentences: int = 2,
) -> list[dict] | None:
    """
    Splits a PDF into fixed-size sentence-aware chunks.

    Cross-page fix: all pages are joined into one string with markers before
    splitting, so spaCy sees the full document and never severs a sentence at
    a page boundary. Page references are recovered from the markers after splitting.

    No per-sentence embeddings — boundary is purely token count.
    Overlap carries the last N sentences into the next chunk so context
    isn't lost at boundaries.
    """
    pages = extract_text_from_pdf(pdf_path)

    if pages is None:
        return None   # already logged inside extract_text_from_pdf

    # ── Build joined document string with page markers ────────────────────────
    # Markers are injected between pages so page references can be recovered
    # after spaCy splits the full document into sentences.
    # Format: "|||PAGE_5|||" — unique enough to never appear in real text.
    parts = []
    for p in pages:
        parts.append(p["text"])
        parts.append(f"|||PAGE_{p['page_num'] + 1}|||")   # marker before next page

    joined_text = " ".join(parts)
    max_sentence_chars = max_chunk_tokens * 4   # consistent with token estimation below

    raw_sentences = split_into_sentences(joined_text, max_sentence_chars)

    # ── Recover page references from markers ──────────────────────────────────
    # Walk through sentences. Track current page using markers encountered.
    # A sentence containing a marker started on the page before that marker.
    sentences = []
    page_refs = []
    current_page = pages[0]["page_num"]
    marker_pattern = re.compile(r'\|\|\|PAGE_(\d+)\|\|\|')

    for sent in raw_sentences:
        marker_match = marker_pattern.search(sent)

        if marker_match:
            # Sentence crossed a page boundary — assign to the page it started on
            page_refs.append(current_page)
            # Advance current page to what the marker indicates
            current_page = int(marker_match.group(1))
            # Clean the marker out of the sentence text before storing
            clean_sent = marker_pattern.sub("", sent).strip()
            if clean_sent and re.search(r'[a-zA-Z0-9]', clean_sent):
                sentences.append(clean_sent)
            else:
                page_refs.pop()   # remove the ref we just added if sentence is empty
        else:
            sentences.append(sent)
            page_refs.append(current_page)

    if not sentences:
        logging.error(f"'{pdf_path}' — text was extracted but no sentences found.")
        return None

    # ── Build chunks ──────────────────────────────────────────────────────────
    chunks = []
    current_sents: list[str] = []
    current_pages: list[int] = []

    def save_chunk():
        if current_sents:
            chunks.append({
                "chunk_id":   len(chunks),
                "text":       " ".join(current_sents),
                "page_start": current_pages[0],
                "page_end":   current_pages[-1],
                "sentences":  len(current_sents),
            })

    for sent, page in zip(sentences, page_refs):
        current_sents.append(sent)
        current_pages.append(page)

        token_est = sum(len(s) for s in current_sents) // 4

        if token_est >= max_chunk_tokens:
            save_chunk()
            current_sents = current_sents[-overlap_sentences:]
            current_pages = current_pages[-overlap_sentences:]

    save_chunk()   # force-flush remaining sentences

    return chunks

In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Claude Contextual Enrichment  (Anthropic blog prompt — verbatim)
# ─────────────────────────────────────────────────────────────────────────────

def add_contextual_retrieval(
    chunks: list[dict],
    whole_document: str,
) -> list[dict]:
    """
    Prepends Claude-generated context to each chunk using the exact prompt
    from the Anthropic Contextual Retrieval blog.

    Fixes applied:
      - Truncates whole_document to 150k chars (~100k tokens) to stay within
        Claude's context window for very large PDFs
      - Exponential backoff retry (up to 3 attempts) on RateLimitError
        so large batches don't crash mid-pipeline
      - Progress counter so you know it hasn't hung
    """
    MAX_DOC_CHARS = 150_000

    if len(whole_document) > MAX_DOC_CHARS:
        print(
            f"  ⚠️  Document is {len(whole_document):,} chars — "
            f"truncating to {MAX_DOC_CHARS:,} for Claude context window"
        )
        whole_document = whole_document[:MAX_DOC_CHARS]

    enriched = []

    for i, chunk in enumerate(chunks):
        # ── Exact prompt from Anthropic blog ──────────────────────────────────
        prompt = (
            "<document>\n"
            f"{whole_document}\n"
            "</document>\n\n"
            "Here is the chunk we want to situate within the whole document\n"
            "<chunk>\n"
            f"{chunk['text']}\n"
            "</chunk>\n\n"
            "Please give a short succinct context to situate this chunk within "
            "the overall document for the purposes of improving search retrieval "
            "of the chunk. Answer only with the succinct context and nothing else."
        )

        # ── Retry with exponential backoff on rate limit ──────────────────────
        response = None
        for attempt in range(3):
            try:
                response = anthropic_client.messages.create(
                    model="claude-haiku-4-5-20251001",
                    max_tokens=100,          # blog: context = 50-100 tokens
                    messages=[{"role": "user", "content": prompt}],
                )
                break
            except RateLimitError:
                wait = 2 ** attempt          # 1s → 2s → 4s
                print(f"  ⏳ Rate limited (attempt {attempt+1}/3), retrying in {wait}s…")
                time.sleep(wait)

        if response is None:
            print(f"  ❌ Failed to enrich chunk {i} after 3 attempts — using raw text")
            enriched.append({
                **chunk,
                "context": "",
                "contextualized_text": chunk["text"],
            })
            continue

        context = response.content[0].text.strip()
        enriched.append({
            **chunk,
            "context": context,
            "contextualized_text": f"{context} {chunk['text']}",
        })
        print(f"  ✅ Chunk {i+1}/{len(chunks)} enriched")

    return enriched


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Folder Runner
# ─────────────────────────────────────────────────────────────────────────────

def process_pdf_folder(
    folder: str = "./RAG_research_paper",
    add_context: bool = True,
) -> dict[str, list[dict]]:
    """
    Walks a folder of PDFs and returns chunked (+ optionally enriched) results.
    Skipped files are tracked and logged — one bad PDF won't kill the batch.
    """
    pdf_files = list(Path(folder).glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in '{folder}'")

    logging.info(f"Found {len(pdf_files)} PDF(s) in '{folder}'")

    results = {}
    skipped_files = []

    for pdf_file in pdf_files:
        logging.info(f"Processing: {pdf_file.name}")

        try:
            chunks = semantic_chunk_pdf(str(pdf_file))

            if chunks is None:
                skipped_files.append(pdf_file.name)
                logging.warning(f"Skipping '{pdf_file.name}' — no chunks produced.")
                continue

            logging.info(f"'{pdf_file.name}' — {len(chunks)} semantic chunks created.")

            if add_context:
                full_text = " ".join(c["text"] for c in chunks)
                logging.info(f"'{pdf_file.name}' — adding contextual retrieval context via Claude.")
                chunks = add_contextual_retrieval(chunks, full_text)

            results[pdf_file.stem] = chunks

        except Exception as e:
            skipped_files.append(pdf_file.name)
            logging.error(f"Skipping '{pdf_file.name}' — unexpected error: {e}")
            continue

    if skipped_files:
        logging.warning(f"Skipped {len(skipped_files)} file(s): {skipped_files}")

    return results
K



In [9]:

# ─────────────────────────────────────────────────────────────────────────────
# Usage
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    all_chunks = process_pdf_folder(
        folder="./RAG_research_paper",
        add_context=True,
    )

    for doc_name, chunks in all_chunks.items():
        print(f"\n{'═'*60}")
        print(f"Document : {doc_name}")
        print(f"Chunks   : {len(chunks)}")
        print(f"{'═'*60}")
        print("Sample contextualized chunk:")
        print(chunks[0]["contextualized_text"][:300])

Found 5 PDF(s) in './RAG_research_paper'

────────────────────────────────────────────────────────────
Processing: 2025.acl-long.131.pdf
────────────────────────────────────────────────────────────
  → 31 semantic chunks created
  Adding contextual retrieval context via Claude…
  ✅ Chunk 1/31 enriched
  ✅ Chunk 2/31 enriched
  ✅ Chunk 3/31 enriched
  ✅ Chunk 4/31 enriched


KeyboardInterrupt: 

In [ ]:
import chromadb
from pathlib import Path

from vectorstore import (
    get_chroma_client,
    get_or_create_collection,
    add_chunks_to_collection,
    collection_stats,
)
from embedding_cache import embed_chunks
from rag_pipeline import (
    semantic_chunk_pdf,
    add_contextual_retrieval,
    process_pdf_folder,
)


# ─────────────────────────────────────────────────────────────────────────────
# Vectorstore Initialisation — handles both existing and new
# ─────────────────────────────────────────────────────────────────────────────

def initialise_vectorstore(
    pdf_folder: str        = "./RAG_research_paper",
    persist_path: str      = "./chroma_db",
    collection_name: str   = "rag_papers",
    cache_path: str        = "./embedding_cache.json",
    add_context: bool      = True,
    force_rebuild: bool    = False,
) -> chromadb.Collection:
    """
    Smart initialisation — checks what's already stored and acts accordingly.

    Case 1 — Collection exists and has chunks:
        Skips everything. Returns existing collection immediately.

    Case 2 — Collection is empty or force_rebuild=True:
        Runs the full pipeline: chunk → contextualise → embed (cached) → store.

    Args:
        pdf_folder:       path to your folder of PDFs
        persist_path:     where ChromaDB saves data on disk
        collection_name:  name of the ChromaDB collection
        cache_path:       path to embedding cache JSON file
        add_context:      whether to run Claude contextual enrichment
        force_rebuild:    set True to wipe and rebuild even if data exists

    Returns:
        collection: ready-to-query ChromaDB collection
    """
    client     = get_chroma_client(persist_path)
    collection = get_or_create_collection(client, collection_name)

    # ── Case 1: collection already has data ───────────────────────────────────
    if collection.count() > 0 and not force_rebuild:
        print(f"\n✅ Vectorstore already exists — skipping pipeline")
        collection_stats(collection)
        return collection

    # ── Case 2: empty or force rebuild ────────────────────────────────────────
    if force_rebuild and collection.count() > 0:
        print(f"\n🔄 force_rebuild=True — wiping '{collection_name}' and rebuilding…")
        client.delete_collection(name=collection_name)
        collection = get_or_create_collection(client, collection_name)

    print(f"\n🚀 Building vectorstore from '{pdf_folder}'…")

    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in '{pdf_folder}'")

    print(f"   Found {len(pdf_files)} PDF(s) to process\n")

    for pdf_file in pdf_files:
        source_name = pdf_file.stem
        print(f"{'─'*60}")
        print(f"  Processing: {pdf_file.name}")
        print(f"{'─'*60}")

        try:
            # Step 1 — Semantic chunking
            chunks = semantic_chunk_pdf(str(pdf_file))
            print(f"  → {len(chunks)} semantic chunks created")

            # Step 2 — Claude contextual enrichment (optional)
            if add_context:
                full_text = " ".join(c["text"] for c in chunks)
                chunks = add_contextual_retrieval(chunks, full_text)

            # Step 3 — Embed chunks (uses cache — skips API for known chunks)
            chunks = embed_chunks(chunks, cache_path=cache_path)

            # Step 4 — Store in ChromaDB
            add_chunks_to_collection(collection, chunks, source_name)

        except Exception as e:
            print(f"  ❌ Skipping '{pdf_file.name}': {e}")
            continue

    print(f"\n{'═'*60}")
    collection_stats(collection)

    return collection

In [ ]:
# First run — builds everything from scratch
collection = initialise_vectorstore(pdf_folder="./RAG_research_paper")

# Every run after — detects existing data, skips pipeline entirely
#collection = initialise_vectorstore(pdf_folder="./RAG_research_paper")

# Force a full rebuild (e.g. after changing chunk size or distance metric)
#collection = initialise_vectorstore(pdf_folder="./RAG_research_paper", force_rebuild=True)